# Vaccination Data Analysis and Visualization
## Google Colab Project

This notebook performs the vaccination-project workflow using the supplied WHO-style datasets:

- **Coverage data**
- **Vaccine Introduction**
- **Vaccine Schedule**

The project specification also calls for **Incidence Rate** and **Reported Cases** tables. If those two files are available, upload them in the upload cell below; the notebook automatically enables the disease-impact analyses.

### Project goals
- Data cleaning and quality checks
- Exploratory data analysis
- Vaccination coverage trends
- Antigen and country comparisons
- First-dose/subsequent-dose and booster analysis where the available fields support it
- Vaccine introduction analysis
- Vaccine schedule analysis
- WHO-region comparisons
- Measles 95% target analysis
- SQL/SQLite database creation
- Power BI-ready exports


In [ ]:
# Install required libraries
!pip -q install pandas openpyxl matplotlib seaborn scipy plotly sqlalchemy

import os
import re
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import pearsonr
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

BASE = Path("/content")
DATA_DIR = BASE / "vaccination_data"
DATA_DIR.mkdir(exist_ok=True)

print("Data directory:", DATA_DIR)


## 1. Upload datasets

The three supplied files can be uploaded directly. If you have the **Incidence Rate** and **Reported Cases** Excel files, upload them at the same time.

Expected filenames can contain:
- `coverage`
- `introduction`
- `schedule`
- `incidence`
- `reported` or `cases`


In [ ]:
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    (DATA_DIR / filename).write_bytes(content)

print("Uploaded files:")
for p in sorted(DATA_DIR.iterdir()):
    print(" -", p.name)


In [ ]:
# Load Excel/CSV files automatically
def read_any(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_excel(path, sheet_name="Data")

def identify_file(keyword_list):
    for p in DATA_DIR.iterdir():
        if p.suffix.lower() not in [".csv", ".xlsx", ".xls"]:
            continue
        name = p.stem.lower().replace("_", "-").replace(" ", "-")
        if all(k in name for k in keyword_list):
            return p
    return None

file_map = {
    "coverage": identify_file(["coverage"]),
    "vaccine_introduction": identify_file(["introduction"]) or identify_file(["intro"]),
    "vaccine_schedule": identify_file(["schedule"]),
    "incidence_rate": identify_file(["incidence"]),
    "reported_cases": identify_file(["reported"]) or identify_file(["cases"]),
}

for k, p in file_map.items():
    print(f"{k:24} -> {p}")

tables = {}
for name, path in file_map.items():
    if path is not None:
        tables[name] = read_any(path)
        print(f"{name}: {tables[name].shape}")


## 2. Data dictionary

The supplied coverage file contains fields such as country code/name, year, antigen, coverage category, target number, doses and coverage. The vaccine-introduction file contains country, WHO region, year, vaccine description and introduction status. The vaccine-schedule file contains vaccine code/description, schedule rounds, target population, geographic area and age administered.

The project specification describes these five source tables and asks for cleaning, SQL, Power BI and documentation.


In [ ]:
def clean_columns(df):
    df = df.copy()
    df.columns = [
        re.sub(r"[^a-z0-9]+", "_", str(c).strip().lower()).strip("_")
        for c in df.columns
    ]
    aliases = {
        "countryname": "country_name",
        "iso_3_code": "code",
        "vaccinecode": "vaccine_code",
        "vaccine_description": "vaccine_description",
        "schedulerounds": "schedule_rounds",
        "targetpop": "target_pop",
        "targetpop_description": "target_pop_description",
        "geoarea": "geo_area",
        "ageadministered": "age_administered",
        "sourcecomment": "source_comment",
        "who_region": "who_region",
        "antigen_description": "antigen_description",
        "coverage_category_description": "coverage_category_description",
        "target_number": "target_number",
        "doses": "doses",
        "coverage": "coverage",
        "incidence_rate": "incidence_rate",
    }
    df = df.rename(columns={c: aliases.get(c, c) for c in df.columns})

    if "year" in df:
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

    for c in ["coverage", "target_number", "doses", "schedule_rounds", "incidence_rate", "cases"]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

tables = {name: clean_columns(df) for name, df in tables.items()}

for name, df in tables.items():
    print(f"\n{name}: {df.shape}")
    print(list(df.columns))


## 3. Data-quality report

In [ ]:
def quality_report(df):
    return pd.DataFrame({
        "data_type": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(dropna=True),
        "duplicate_count": df.duplicated().sum()
    }).sort_values("missing_percent", ascending=False)

for name, df in tables.items():
    print("\n###", name)
    display(quality_report(df).head(30))


In [ ]:
# Cleaning and validation
def clean_table(df):
    df = df.copy().drop_duplicates()

    # Trim text
    for c in df.select_dtypes(include="object").columns:
        df[c] = df[c].astype("string").str.strip()

    # Coverage should be 0-100 when supplied
    if "coverage" in df:
        invalid = (df["coverage"] < 0) | (df["coverage"] > 100)
        print("Invalid coverage values:", int(invalid.fillna(False).sum()))
        df.loc[invalid, "coverage"] = np.nan

    # Counts should not be negative
    for c in ["target_number", "doses", "cases"]:
        if c in df:
            invalid = df[c] < 0
            print(f"Invalid negative {c} values:", int(invalid.fillna(False).sum()))
            df.loc[invalid, c] = np.nan

    return df

tables = {name: clean_table(df) for name, df in tables.items()}
print("Cleaning completed.")


## 4. Vaccination coverage analysis

In [ ]:
coverage = tables.get("coverage")

if coverage is None:
    raise ValueError("Coverage data is required.")

print("Coverage rows:", len(coverage))
display(coverage.head())


In [ ]:
# Overall coverage trend
if {"year", "coverage"}.issubset(coverage.columns):
    yearly = (
        coverage.dropna(subset=["coverage"])
        .groupby("year", as_index=False)["coverage"]
        .mean()
    )

    display(yearly.tail(20))

    plt.figure(figsize=(12, 5))
    plt.plot(yearly["year"], yearly["coverage"], marker="o")
    plt.axhline(95, linestyle="--", label="95% reference")
    plt.title("Average Vaccination Coverage by Year")
    plt.xlabel("Year")
    plt.ylabel("Coverage (%)")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


In [ ]:
# Coverage by antigen
if {"antigen_description", "coverage"}.issubset(coverage.columns):
    antigen_summary = (
        coverage.dropna(subset=["coverage"])
        .groupby("antigen_description", as_index=False)
        .agg(
            average_coverage=("coverage", "mean"),
            records=("coverage", "count")
        )
        .sort_values("average_coverage", ascending=False)
    )

    display(antigen_summary.head(25))

    top = antigen_summary.head(15).sort_values("average_coverage")
    plt.figure(figsize=(11, 7))
    plt.barh(top["antigen_description"], top["average_coverage"])
    plt.xlabel("Average coverage (%)")
    plt.title("Average Coverage by Antigen")
    plt.show()


In [ ]:
# Country coverage
country_cols = [c for c in ["code", "name", "country_name"] if c in coverage.columns]

if country_cols and "coverage" in coverage:
    country_summary = (
        coverage.dropna(subset=["coverage"])
        .groupby(country_cols, as_index=False)
        .agg(
            average_coverage=("coverage", "mean"),
            records=("coverage", "count")
        )
        .sort_values("average_coverage")
    )

    print("Lowest average coverage:")
    display(country_summary.head(20))

    print("Highest average coverage:")
    display(country_summary.tail(20).sort_values("average_coverage", ascending=False))


## 5. Dose drop-off and booster analysis

The coverage dataset includes antigen descriptions such as booster doses. The notebook uses those descriptions when they explicitly identify a dose/booster. A true first-dose-to-second-dose drop-off is calculated only when comparable dose indicators are available.


In [ ]:
# Identify dose/booster antigens from descriptions
if "antigen_description" in coverage:
    dose_map = coverage[
        coverage["antigen_description"].astype(str).str.contains(
            r"1st|2nd|3rd|4th|5th|dose|booster",
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    dose_summary = (
        dose_map.groupby("antigen_description", as_index=False)["coverage"]
        .mean()
        .sort_values("average" if "average" in dose_map else "coverage", ascending=False)
    )

    display(dose_summary.head(50))


In [ ]:
# Explicit booster coverage trend
if "antigen_description" in coverage:
    booster = coverage[
        coverage["antigen_description"].astype(str).str.contains(
            "booster|2nd booster|1st booster",
            case=False,
            na=False
        )
    ]

    if len(booster) and {"year", "coverage"}.issubset(booster.columns):
        booster_trend = booster.groupby("year", as_index=False)["coverage"].mean()
        display(booster_trend)

        plt.figure(figsize=(11, 5))
        plt.plot(booster_trend["year"], booster_trend["coverage"], marker="o")
        plt.title("Booster-Dose Coverage Over Time")
        plt.xlabel("Year")
        plt.ylabel("Average coverage (%)")
        plt.grid(alpha=0.2)
        plt.show()
    else:
        print("No explicit booster records were identified.")


## 6. WHO-region analysis

The vaccine introduction and vaccine schedule tables contain WHO-region fields. Coverage itself can be joined to those tables by country code and year when the fields are available.


In [ ]:
intro = tables.get("vaccine_introduction")
schedule = tables.get("vaccine_schedule")

if intro is not None:
    display(
        intro.groupby("who_region", dropna=False)
        .size()
        .reset_index(name="records")
        .sort_values("records", ascending=False)
    )

if schedule is not None:
    display(
        schedule.groupby(["who_region", "year"], dropna=False)
        .size()
        .reset_index(name="schedule_records")
        .sort_values(["year", "schedule_records"], ascending=[False, False])
        .head(50)
    )


## 7. Vaccine introduction analysis

In [ ]:
if intro is not None:
    print("Introduction status:")
    display(intro["intro"].value_counts(dropna=False))

    # Earliest introduction year for each country/vaccine
    if {"code", "description", "year", "intro"}.issubset(intro.columns):
        introduced = intro[
            intro["intro"].astype(str).str.lower().isin(["yes", "y", "true", "1"])
        ].copy()

        first_intro = (
            introduced.groupby(["code", "description"], as_index=False)["year"]
            .min()
            .rename(columns={"year": "first_introduction_year"})
        )

        display(first_intro.head(50))

        region_intro = (
            introduced.groupby(["who_region", "year"], as_index=False)
            .size()
            .rename(columns={"size": "introduced_records"})
        )

        fig = px.line(
            region_intro,
            x="year",
            y="introduced_records",
            color="who_region",
            markers=True,
            title="Vaccine Introduction Records by WHO Region"
        )
        fig.show()


## 8. Vaccine schedule analysis

In [ ]:
if schedule is not None:
    # Schedule rounds
    if "schedule_rounds" in schedule:
        round_summary = (
            schedule.groupby("schedule_rounds", dropna=False)
            .size()
            .reset_index(name="records")
            .sort_values("schedule_rounds")
        )
        display(round_summary)

        plt.figure(figsize=(10, 5))
        plt.bar(
            round_summary["schedule_rounds"].astype(str),
            round_summary["records"]
        )
        plt.title("Vaccine Schedule Rounds")
        plt.xlabel("Schedule round")
        plt.ylabel("Records")
        plt.show()

    # Target population
    if "target_pop_description" in schedule:
        display(
            schedule["target_pop_description"]
            .value_counts(dropna=False)
            .head(20)
            .to_frame("records")
        )

    # Age administered
    if "age_administered" in schedule:
        display(
            schedule["age_administered"]
            .value_counts(dropna=False)
            .head(30)
            .to_frame("records")
        )


## 9. Disease incidence and reported cases

If the Incidence Rate and Reported Cases files were uploaded, this section automatically performs the cross-table analysis requested in the project.


In [ ]:
incidence = tables.get("incidence_rate")
cases = tables.get("reported_cases")

if incidence is None:
    print("Incidence Rate file not uploaded. Disease-incidence analysis will be skipped.")
else:
    print("Incidence data:", incidence.shape)
    display(incidence.head())

if cases is None:
    print("Reported Cases file not uploaded. Reported-case analysis will be skipped.")
else:
    print("Reported Cases data:", cases.shape)
    display(cases.head())


In [ ]:
# Merge coverage with incidence/cases at country-year level
analysis = coverage.copy()

keys = [c for c in ["code", "name", "country_name", "year"] if c in analysis.columns]

# Prefer code + year where possible
if "code" in analysis.columns and "year" in analysis.columns:
    keys = ["code", "year"]

if incidence is not None and "incidence_rate" in incidence and set(keys).issubset(incidence.columns):
    ir = incidence.groupby(keys, as_index=False)["incidence_rate"].mean()
    analysis = analysis.merge(ir, on=keys, how="left")

if cases is not None and "cases" in cases and set(keys).issubset(cases.columns):
    ca = cases.groupby(keys, as_index=False)["cases"].sum()
    analysis = analysis.merge(ca, on=keys, how="left")

print("Combined analysis table:", analysis.shape)
display(analysis.head())


In [ ]:
# Correlation analysis
for outcome in ["incidence_rate", "cases"]:
    if outcome in analysis.columns and "coverage" in analysis.columns:
        d = analysis[["coverage", outcome]].dropna()

        if len(d) >= 3:
            r, p = pearsonr(d["coverage"], d[outcome])
            print(f"Coverage vs {outcome}: Pearson r = {r:.4f}, p = {p:.4g}, n = {len(d):,}")

            fig = px.scatter(
                d,
                x="coverage",
                y=outcome,
                trendline="ols",
                title=f"Vaccination Coverage vs {outcome}"
            )
            fig.show()
        else:
            print(f"Not enough paired observations for {outcome}.")


In [ ]:
# Low-coverage / high-incidence identification
if {"coverage", "incidence_rate"}.issubset(analysis.columns):
    low_threshold = 80
    incidence_threshold = analysis["incidence_rate"].median()

    high_incidence_low_coverage = analysis[
        (analysis["coverage"] < low_threshold) &
        (analysis["incidence_rate"] > incidence_threshold)
    ].copy()

    cols = [
        c for c in [
            "code", "name", "country_name", "year",
            "antigen_description", "coverage", "incidence_rate"
        ] if c in high_incidence_low_coverage.columns
    ]

    display(
        high_incidence_low_coverage[cols]
        .sort_values("incidence_rate", ascending=False)
        .head(50)
    )


## 10. Measles coverage and the 95% target

In [ ]:
measles = coverage.copy()

if "antigen_description" in measles:
    measles = measles[
        measles["antigen_description"].astype(str).str.contains(
            "measles",
            case=False,
            na=False
        )
    ]

if len(measles) and {"year", "coverage"}.issubset(measles.columns):
    measles_year = (
        measles.dropna(subset=["coverage"])
        .groupby("year", as_index=False)["coverage"]
        .mean()
    )
    measles_year["target_95"] = 95
    measles_year["gap_to_95"] = 95 - measles_year["coverage"]

    display(measles_year)

    fig = px.line(
        measles_year,
        x="year",
        y=["coverage", "target_95"],
        markers=True,
        title="Measles Vaccination Coverage vs 95% Target"
    )
    fig.show()
else:
    print("No measles coverage records were found.")


## 11. SQLite database and SQL analysis

In [ ]:
DB_PATH = BASE / "vaccination_project.db"
conn = sqlite3.connect(DB_PATH)

for name, df in tables.items():
    df.to_sql(name, conn, if_exists="replace", index=False)

print("SQLite database created:", DB_PATH)
print("\nTables:")
for name in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM [{name}]").fetchone()[0]
    print(f" - {name}: {count:,} rows")


In [ ]:
# Example SQL queries
queries = {}

if "coverage" in tables:
    queries["Annual average coverage"] = '''
        SELECT year, AVG(coverage) AS average_coverage
        FROM coverage
        WHERE coverage IS NOT NULL
        GROUP BY year
        ORDER BY year;
    '''

    queries["Coverage by antigen"] = '''
        SELECT antigen_description,
               AVG(coverage) AS average_coverage,
               COUNT(*) AS records
        FROM coverage
        WHERE coverage IS NOT NULL
        GROUP BY antigen_description
        ORDER BY average_coverage DESC;
    '''

if "vaccine_introduction" in tables:
    queries["Introductions by WHO region and year"] = '''
        SELECT who_region, year, COUNT(*) AS records
        FROM vaccine_introduction
        WHERE LOWER(intro) IN ('yes','y','true','1')
        GROUP BY who_region, year
        ORDER BY year, who_region;
    '''

for title, query in queries.items():
    print("\n###", title)
    display(pd.read_sql_query(query, conn))


## 12. Power BI-ready exports

In [ ]:
EXPORT_DIR = BASE / "powerbi_export"
EXPORT_DIR.mkdir(exist_ok=True)

for name, df in tables.items():
    path = EXPORT_DIR / f"{name}_clean.csv"
    df.to_csv(path, index=False)
    print(path)

# Combined analytical table, when available
analysis_path = EXPORT_DIR / "coverage_disease_analysis.csv"
analysis.to_csv(analysis_path, index=False)
print(analysis_path)


## 13. Answering the project questions

### Questions supported by the supplied tables
- How do vaccination rates correlate with disease incidence? → calculated when Incidence Rate is uploaded.
- What is the drop-off between doses? → analyzed where dose/booster identifiers are present.
- Has booster uptake increased over time? → booster trend where explicitly identifiable.
- Which regions have high disease incidence despite high vaccination rates? → calculated when incidence data is uploaded.
- Is there a relationship between vaccine introduction and disease cases? → requires Reported Cases data.
- What percentage of the target population is covered by each vaccine? → antigen-level coverage analysis.
- How does the vaccination schedule relate to target population coverage? → schedule-round and target-population analysis.
- Are introduction timelines different across WHO regions? → WHO-region introduction analysis.
- What are the measles coverage gaps against 95%? → calculated above.

### Data limitations
The five project tables do **not** necessarily contain gender, education, urban/rural, population density, socioeconomic status, month/season, vaccination strategy, or all high-risk age-group variables. Those questions should not be answered unless the corresponding variables are actually present.


## 14. Final project outputs

After running the notebook, the main deliverables are:

1. Cleaned datasets
2. EDA tables and charts
3. Correlation analysis where disease data is supplied
4. SQLite database: `vaccination_project.db`
5. Power BI CSV files in `powerbi_export/`
6. Analytical tables for dashboard development

### Recommended Power BI pages
1. Executive Overview
2. Vaccination Coverage
3. Disease Incidence & Reported Cases
4. Vaccine Introduction
5. Vaccine Schedule
6. Measles 95% Target

**Important:** Correlation and before/after comparisons are descriptive analyses and should not be interpreted as causal vaccine-effectiveness estimates without an appropriate epidemiological study design.
